# Urban Traffic Bonn
Simulate the urban traffic from the city of Bonn using [ArcGIS API for Python](https://developers.arcgis.com/python/).

In [1]:
# author: Jan Tschada
# SPDX-License-Identifer: Apache-2.0

## Required Python modules
You need to install ArcGIS API for Python, please follow the [guide](https://developers.arcgis.com/python/guide/anaconda/).

In [2]:
from arcgis.gis import GIS
from arcgis.features import FeatureSet
from datetime import date, datetime, timedelta
from georapid.client import GeoRapidClient
from georapid.factory import EnvironmentClientFactory
from geourban.services import aggregate, query, simulations, top
from geourban.types import GridType, VehicleType

## Connect to ArcGIS Online

In [3]:
gis = GIS()

## Create a map showing the city of Bonn

In [6]:
bonn_map = gis.map('Bonn, Germany')
bonn_map.zoom = 13
bonn_map

Map(center=[6574418.690342708, 790277.8818862307], extent={'xmin': 781149.6836411823, 'ymin': 6560008.89814410…

# Connect to geourban services
The host parameter must target the specific host like "geoprotests.p.rapidapi.com". Furthermore, the factory directly access `os.environ['x_rapidapi_key']` and uses the specified API key as a header parameter. Otherwise, `georapid.factory.EnvironmentClientFactory.create_client_with_host()` will raise a `ValueError`.

Please, check out the [RapidAPI Account Creation and Management Guide](https://docs.rapidapi.com/docs/account-creation-and-settings).

In [7]:
client = EnvironmentClientFactory.create_client_with_host('geourban.p.rapidapi.com')

# List the available simulations

In [8]:
urban_simulations = simulations(client)
urban_simulations

[{'region': 'DEA2D', 'name': 'Aachen, Stadt', 'date': '2023-12-10'},
 {'region': 'USQ27768421',
  'name': 'Allegiant Stadium, Las Vegas',
  'date': '2024-02-11'},
 {'region': 'DE246', 'name': 'Bayreuth, Landkreis', 'date': '2023-12-28'},
 {'region': 'DE300', 'name': 'Berlin', 'date': '2023-08-25'},
 {'region': 'DEA22', 'name': 'Bonn, Kreisfreie Stadt', 'date': '2023-08-24'},
 {'region': 'DEE01', 'name': 'Dessau-Roßlau, Stadt', 'date': '2023-08-24'},
 {'region': 'DED21', 'name': 'Dresden, Stadt', 'date': '2023-12-10'},
 {'region': 'DEA12',
  'name': 'Duisburg, Kreisfreie Stadt',
  'date': '2023-12-22'},
 {'region': 'DE113', 'name': 'Esslingen, Landkreis', 'date': '2023-12-19'},
 {'region': 'ITI14', 'name': 'Firenze', 'date': '2024-01-07'},
 {'region': 'DE712',
  'name': 'Frankfurt am Main, Kreisfreie Stadt',
  'date': '2023-11-21'},
 {'region': 'DE929', 'name': 'Hannover, Region', 'date': '2023-11-09'},
 {'region': 'DE115', 'name': 'Ludwigsburg, Landkreis', 'date': '2023-12-18'},
 {'reg

## Request the top 5 accumulated car traffic grid cells
We request these hotspots for the city of Bonn by using the urban region code `DEA22`, the simulation date `2023-08-24`, the vehicle type `Car`, and the grid type `agent`. The returned GeoJSON features represents the grid cells with the highest car throughput.

In [9]:
bonn_region_code = 'DEA22'
simulation_date = date(2023, 8, 24)
vehicle_type = VehicleType.CAR
grid_type = GridType.AGENT
limit = 5
top_traffic_grid_cells = top(client, bonn_region_code, simulation_date, vehicle_type, grid_type, limit=limit)
top_traffic_grid_cells

{'type': 'FeatureCollection',
 'features': [{'type': 'Feature',
   'geometry': {'type': 'Polygon',
    'coordinates': [[[7.07375, 50.74601],
      [7.07504, 50.74601],
      [7.07569, 50.74672],
      [7.07504, 50.74743],
      [7.07375, 50.74743],
      [7.0731, 50.74672],
      [7.07375, 50.74601]]]},
   'properties': {'start_time': '2023-08-24T08:00:00',
    'end_time': '2023-08-24T08:59:59',
    'agent_count': 439}},
  {'type': 'Feature',
   'geometry': {'type': 'Polygon',
    'coordinates': [[[7.07375, 50.74601],
      [7.07504, 50.74601],
      [7.07569, 50.74672],
      [7.07504, 50.74743],
      [7.07375, 50.74743],
      [7.0731, 50.74672],
      [7.07375, 50.74601]]]},
   'properties': {'start_time': '2023-08-24T09:00:00',
    'end_time': '2023-08-24T09:59:59',
    'agent_count': 427}},
  {'type': 'Feature',
   'geometry': {'type': 'Polygon',
    'coordinates': [[[7.07375, 50.74601],
      [7.07504, 50.74601],
      [7.07569, 50.74672],
      [7.07504, 50.74743],
      [7.073

## Convert the returned GeoJSON result into a FeatureSet
The FeatureSet offers direct access to a spatially enabled dataframe. We can easily inspect the time frames (`start_time` - `end_time`) and the number of car vehicles `agent_count`.

In [10]:
top_traffic_fset = FeatureSet.from_geojson(top_traffic_grid_cells)
top_traffic_sdf = top_traffic_fset.sdf
top_traffic_sdf

,start_time,end_time,agent_count,OBJECTID,SHAPE
0,2023-08-24T08:00:00,2023-08-24T08:59:59,439,1,"{""rings"": [[[7.07375, 50.74601], [7.07504, 50...."
1,2023-08-24T09:00:00,2023-08-24T09:59:59,427,2,"{""rings"": [[[7.07375, 50.74601], [7.07504, 50...."
2,2023-08-24T07:00:00,2023-08-24T07:59:59,422,3,"{""rings"": [[[7.07375, 50.74601], [7.07504, 50...."
3,2023-08-24T08:00:00,2023-08-24T08:59:59,421,4,"{""rings"": [[[7.07569, 50.74672], [7.07699, 50...."
4,2023-08-24T09:00:00,2023-08-24T09:59:59,408,5,"{""rings"": [[[7.07569, 50.74672], [7.07699, 50...."


## Map the traffic grid cells

In [11]:
top_traffic_sdf.spatial.plot(bonn_map, renderer_type='s', colors='#E80000', alpha=0.3)
bonn_map

Map(center=[7.099187000000029, 50.734316777071584], extent={'spatialReference': {'latestWkid': 3857, 'wkid': 1…

# Query the simulated agents nearby
We are using the center of this crossroad intersection, request the simulated agents being within a distance of `250 meters`, and specify a `30 seconds` time window starting from `08:00:00`.

In [12]:
simulation_datetime = datetime.fromisoformat('2023-08-24T08:00:00')
(latitude, longitude) = (50.746708, 7.074405)
(seconds, meters) = (30, 250)
car_positions = query(client, simulation_datetime, vehicle_type, latitude, longitude, seconds, meters)
car_positions_fset = FeatureSet.from_geojson(car_positions)
car_positions_sdf = car_positions_fset.sdf
car_positions_sdf

,trip,person,trip_time,OBJECTID,SHAPE
0,75,46,2023-08-24T08:00:00,1,"{""x"": 7.0737, ""y"": 50.74581, ""spatialReference..."
1,3736,2333,2023-08-24T08:00:00,2,"{""x"": 7.07429, ""y"": 50.74448, ""spatialReferenc..."
2,3485,2168,2023-08-24T08:00:00,3,"{""x"": 7.07189, ""y"": 50.74697, ""spatialReferenc..."
3,1365,845,2023-08-24T08:00:00,4,"{""x"": 7.07359, ""y"": 50.74461, ""spatialReferenc..."
4,3736,2333,2023-08-24T08:00:01,5,"{""x"": 7.07404, ""y"": 50.74448, ""spatialReferenc..."
...,...,...,...,...,...
220,6476,4048,2023-08-24T08:00:29,221,"{""x"": 7.07133, ""y"": 50.74749, ""spatialReferenc..."
221,3485,2168,2023-08-24T08:00:29,222,"{""x"": 7.07328, ""y"": 50.74503, ""spatialReferenc..."
222,3736,2333,2023-08-24T08:00:29,223,"{""x"": 7.07496, ""y"": 50.7459, ""spatialReference..."
223,1365,845,2023-08-24T08:00:29,224,"{""x"": 7.07433, ""y"": 50.74616, ""spatialReferenc..."


## Do some data engineering

We are interested in the movements of a specific agent and want to narrow down its movement behavior.
The GeoJSON properties are treated as strings. So that we should convert the `trip` and `person` columns to `numeric`. Furthermore, we convert the `trip_time` to `datetime`.

In [13]:
import pandas as pd

In [14]:
car_positions_sdf[['trip', 'person']] = car_positions_sdf[['trip', 'person']].apply(pd.to_numeric)
car_positions_sdf[['trip_time']] = car_positions_sdf[['trip_time']].apply(pd.to_datetime)
car_positions_sdf

,trip,person,trip_time,OBJECTID,SHAPE
0,75,46,2023-08-24 08:00:00,1,"{""x"": 7.0737, ""y"": 50.74581, ""spatialReference..."
1,3736,2333,2023-08-24 08:00:00,2,"{""x"": 7.07429, ""y"": 50.74448, ""spatialReferenc..."
2,3485,2168,2023-08-24 08:00:00,3,"{""x"": 7.07189, ""y"": 50.74697, ""spatialReferenc..."
3,1365,845,2023-08-24 08:00:00,4,"{""x"": 7.07359, ""y"": 50.74461, ""spatialReferenc..."
4,3736,2333,2023-08-24 08:00:01,5,"{""x"": 7.07404, ""y"": 50.74448, ""spatialReferenc..."
...,...,...,...,...,...
220,6476,4048,2023-08-24 08:00:29,221,"{""x"": 7.07133, ""y"": 50.74749, ""spatialReferenc..."
221,3485,2168,2023-08-24 08:00:29,222,"{""x"": 7.07328, ""y"": 50.74503, ""spatialReferenc..."
222,3736,2333,2023-08-24 08:00:29,223,"{""x"": 7.07496, ""y"": 50.7459, ""spatialReference..."
223,1365,845,2023-08-24 08:00:29,224,"{""x"": 7.07433, ""y"": 50.74616, ""spatialReferenc..."


In [15]:
person_46_positions_sdf = car_positions_sdf[car_positions_sdf['person']==46]
person_46_positions_sdf

,trip,person,trip_time,OBJECTID,SHAPE
0,75,46,2023-08-24 08:00:00,1,"{""x"": 7.0737, ""y"": 50.74581, ""spatialReference..."
5,75,46,2023-08-24 08:00:01,6,"{""x"": 7.07393, ""y"": 50.74601, ""spatialReferenc..."
8,75,46,2023-08-24 08:00:02,9,"{""x"": 7.07417, ""y"": 50.74621, ""spatialReferenc..."
17,75,46,2023-08-24 08:00:03,18,"{""x"": 7.07441, ""y"": 50.74641, ""spatialReferenc..."
20,75,46,2023-08-24 08:00:04,21,"{""x"": 7.07466, ""y"": 50.7466, ""spatialReference..."
29,75,46,2023-08-24 08:00:05,30,"{""x"": 7.07488, ""y"": 50.7468, ""spatialReference..."
32,75,46,2023-08-24 08:00:06,33,"{""x"": 7.07515, ""y"": 50.74698, ""spatialReferenc..."
42,75,46,2023-08-24 08:00:07,43,"{""x"": 7.07542, ""y"": 50.74716, ""spatialReferenc..."
46,75,46,2023-08-24 08:00:08,47,"{""x"": 7.0757, ""y"": 50.74734, ""spatialReference..."
53,75,46,2023-08-24 08:00:09,54,"{""x"": 7.076, ""y"": 50.74751, ""spatialReference""..."


## Map the car positions nearby

In [16]:
bonn_map = gis.map('Bonn, Germay')
bonn_map.zoom_to_layer(car_positions_sdf)
car_positions_sdf.spatial.plot(bonn_map, renderer_type='s', colors='#E80000', marker_size=7, alpha=0.3)
bonn_map

Map(center=[6574418.690342708, 790277.8818862307], extent={'xmin': 787145.685373777, 'ymin': 6576219.04863025,…

In [24]:
bonn_map = gis.map('Bonn, Germay')
bonn_map.zoom_to_layer(person_46_positions_sdf.copy())
person_46_positions_sdf.spatial.plot(bonn_map, renderer_type='s', colors='#E80000', marker_size=7, alpha=0.3)
bonn_map

Map(center=[6574418.690342708, 790277.8818862307], extent={'xmin': 787440.6820243793, 'ymin': 6576453.02732970…

## Accumulate the speed of car traffic between 08:00 and 09:00 AM

In [18]:
bonn_region_code = 'DEA22'
simulation_datetime = datetime.fromisoformat('2023-08-24T08:00:00')
vehicle_type = VehicleType.CAR
grid_type = GridType.SPEED
car_speed_cells = aggregate(client, bonn_region_code, simulation_datetime, vehicle_type, grid_type)
car_speed_fset = FeatureSet.from_geojson(car_speed_cells)
car_speed_sdf = car_speed_fset.sdf
car_speed_sdf

,start_time,end_time,speed_mean,OBJECTID,SHAPE
0,2023-08-24T08:00:00,2023-08-24T08:59:59,91.055357,1,"{""rings"": [[[7.13598, 50.76448], [7.13728, 50...."
1,2023-08-24T08:00:00,2023-08-24T08:59:59,30.5,2,"{""rings"": [[[7.16321, 50.66779], [7.16451, 50...."
2,2023-08-24T08:00:00,2023-08-24T08:59:59,29.789474,3,"{""rings"": [[[7.16321, 50.66921], [7.16451, 50...."
3,2023-08-24T08:00:00,2023-08-24T08:59:59,29.157895,4,"{""rings"": [[[7.16321, 50.68202], [7.16451, 50...."
4,2023-08-24T08:00:00,2023-08-24T08:59:59,28.944444,5,"{""rings"": [[[7.16321, 50.68344], [7.16451, 50...."
...,...,...,...,...,...
2211,2023-08-24T08:00:00,2023-08-24T08:59:59,70.0,2212,"{""rings"": [[[7.1282, 50.70763], [7.1295, 50.70..."
2212,2023-08-24T08:00:00,2023-08-24T08:59:59,70.0,2213,"{""rings"": [[[7.19044, 50.65782], [7.19174, 50...."
2213,2023-08-24T08:00:00,2023-08-24T08:59:59,70.0,2214,"{""rings"": [[[7.14182, 50.69411], [7.14312, 50...."
2214,2023-08-24T08:00:00,2023-08-24T08:59:59,70.0,2215,"{""rings"": [[[7.13793, 50.69838], [7.13923, 50...."


## Map the accumulated speed of car traffic

In [20]:
bonn_map = gis.map('Bonn, Germay')
bonn_map.zoom = 12
car_speed_sdf.spatial.plot(bonn_map, renderer_type='c', method='esriClassifyNaturalBreaks', class_count=5, col='speed_mean', cmap='RdYlGn')
bonn_map

Map(center=[6574418.690342708, 790277.8818862307], extent={'xmin': 781149.6836411823, 'ymin': 6560008.89814410…

Convert the `start_time`, `end_time` to `datetime` and the `speed_mean` to `numeric`. We restrict the traffic grid cells using a `speed_mean` above `70 km/h`.

In [23]:
car_speed_sdf[['start_time', 'end_time']] = car_speed_sdf[['start_time', 'end_time']].apply(pd.to_datetime)
car_speed_sdf[['speed_mean']] = car_speed_sdf[['speed_mean']].apply(pd.to_numeric)
high_car_speed_sdf = car_speed_sdf[car_speed_sdf['speed_mean'] > 70].copy()
bonn_map = gis.map('Bonn, Germay')
bonn_map.zoom = 12
high_car_speed_sdf.spatial.plot(bonn_map, renderer_type='c', method='esriClassifyNaturalBreaks', class_count=5, col='speed_mean', cmap='RdYlGn')
bonn_map

Map(center=[6574418.690342708, 790277.8818862307], extent={'xmin': 781149.6836411823, 'ymin': 6560008.89814410…